In [1]:
#pip install rdflib owlready2 -q

In [2]:
#pip install  graphviz pydotplus -q

In [3]:
#!apt-get install -y openjdk-11-jdk   -qq

In [4]:
import pandas as pd
import os
from owlready2 import *

## Загрузка данных

In [5]:
# Загружаем данные из CSV файлов
def load_data():
    knowledge_df = pd.read_csv('ontology_data/knowledge.csv')
    competencies_df = pd.read_csv('ontology_data/competencies.csv')
    competency_knowledge_df = pd.read_csv('ontology_data/competency_knowledge.csv')
    positions_df = pd.read_csv('ontology_data/positions.csv')
    position_competency_df = pd.read_csv('ontology_data/position_competency.csv')
    employees_df = pd.read_csv('ontology_data/employees.csv')

    return knowledge_df, competencies_df, competency_knowledge_df, positions_df, position_competency_df, employees_df

In [6]:
knowledge_df, competencies_df, competency_knowledge_df, positions_df, position_competency_df, employees_df = load_data()

In [7]:
knowledge_df

,knowledge_id,knowledge_name
0,1,Математическая статистика
1,2,Линейная алгебра
2,3,Теория вероятностей
3,4,Python программирование
4,5,"Библиотеки ML (scikit-learn, TensorFlow, PyTorch)"
5,6,Обработка естественного языка (NLP)
6,7,Компьютерное зрение
7,8,"Работа с большими данными (Spark, Hadoop)"
8,9,SQL и базы данных
9,10,Визуализация данных


# Создание новой онтологии

## Итерация 1.
Создадим знания и компетенции

In [8]:
# Создаем онтологию
onto = get_ontology("http://test.org/ml_specialist_v1.owl")

with onto:
    class Знание(Thing):
        pass

    class Компетенция(Thing):
        pass

    class требует_знание(ObjectProperty):
        domain = [Компетенция]
        range = [Знание]

    class название(DataProperty):
        domain = [Thing]
        range = [str]

### Создание индивидуальностей для знаний и компетенций



In [9]:
def create_knowledge(knowledge_df):
  with onto:

    knowledge_instances = {}
    for _, row in knowledge_df.iterrows():
        knowledge = Знание(f"знание_{row['knowledge_id']}")
        knowledge.название = [row['knowledge_name']]
        knowledge_instances[row['knowledge_id']] = knowledge
    return knowledge_instances

def create_competencies( competencies_df):
  with onto:
    competency_instances = {}
    for _, row in competencies_df.iterrows():
        competency = Компетенция(f"компетенция_{row['competency_id']}")
        competency.название = [row['competency_name']]
        competency_instances[row['competency_id']] = competency
    return competency_instances


In [10]:
knowledges = create_knowledge(knowledge_df)

In [11]:
knowledges

{1: ml_specialist_v1.знание_1,
 2: ml_specialist_v1.знание_2,
 3: ml_specialist_v1.знание_3,
 4: ml_specialist_v1.знание_4,
 5: ml_specialist_v1.знание_5,
 6: ml_specialist_v1.знание_6,
 7: ml_specialist_v1.знание_7,
 8: ml_specialist_v1.знание_8,
 9: ml_specialist_v1.знание_9,
 10: ml_specialist_v1.знание_10}

In [12]:
for id, item in knowledges.items():
  print (item.название[0])

Математическая статистика
Линейная алгебра
Теория вероятностей
Python программирование
Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Обработка естественного языка (NLP)
Компьютерное зрение
Работа с большими данными (Spark, Hadoop)
SQL и базы данных
Визуализация данных


In [13]:
competencies = create_competencies( competencies_df)
for id, item in competencies.items():
  print (item.название[0])

Математическая подготовка
Программирование и алгоритмы
Машинное обучение
Глубокое обучение
Обработка и анализ данных
Работа с большими данными
Визуализация и представление данных
Развертывание ML-моделей


### Создание отношений между знаниями и компетенциями

In [14]:
# Связываем компетенции со знаниями
def  relate_competencies_knowledge (competency_knowledge_df,  competency_instances, knowledge_instances):
  with onto:
    for _, row in competency_knowledge_df.iterrows():
        competency = competency_instances[row['competency_id']]
        knowledge = knowledge_instances[row['knowledge_id']]
        competency.требует_знание.append(knowledge)


In [15]:
relate_competencies_knowledge (competency_knowledge_df, competencies, knowledges)

Выведем связи между компенциями и знаниями

In [16]:
def print_competency_knowledge():
  print("\n" + "="*60)
  print("СВЯЗИ ЗНАНИЙ И КОМПЕТЕНЦИЙ")
  print("="*60)

  for competency_id, competency in competencies.items():
      comp_name = competency.название[0]
      print(f"\n{comp_name} требует знания:")

      required_knowledge = list(competency.требует_знание)
      for know in required_knowledge:
          print(f"  - {know.название[0]}")

print_competency_knowledge()



СВЯЗИ ЗНАНИЙ И КОМПЕТЕНЦИЙ

Математическая подготовка требует знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей

Программирование и алгоритмы требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Машинное обучение требует знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Глубокое обучение требует знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Обработка и анализ данных требует знания:
  - Математическая статистика
  - SQL и базы данных

Работа с большими данными требует знания:
  - Работа с большими данными (Spark, Hadoop)
  - SQL и базы данных

Визуализация и представление данных требует знания:
  - Визуализация данных
  - Python программирование

Развертывание ML-моделей требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, 

# Итерация 2

Добавление новых концептов в онтологию


In [17]:
with onto:
  class Специалист(Thing):
        pass

  class Должность(Thing):
        pass

  class обладает_знанием(ObjectProperty):
        domain = [Специалист]
        range = [Знание]

  class обладает_компетенцией(ObjectProperty):
        domain = [Специалист]
        range = [Компетенция]

  class может_занимать_должность(ObjectProperty):
        domain = [Специалист]
        range = [Должность]

  class требует_компетенцию(ObjectProperty):
        domain = [Должность]
        range = [Компетенция]

  class имя_сотрудника(DataProperty):
        domain = [Специалист]
        range = [str]

In [18]:
employees_df.groupby(['employee_id', 'full_name'])['knowledge_id'].apply(list)

employee_id  full_name                     
1            Иванов Алексей Сергеевич           [1, 4, 5]
2            Петрова Мария Владимировна         [1, 4, 5]
3            Сидоров Дмитрий Петрович           [1, 2, 9]
4            Козлова Анна Игоревна              [5, 6, 7]
5            Федоров Максим Андреевич           [5, 6, 7]
6            Николаева Екатерина Дмитриевна    [4, 5, 10]
7            Орлов Сергей Викторович            [1, 2, 3]
Name: knowledge_id, dtype: object

### Создание индивидуальностей для сотрудников и их отношений с знаниями


In [19]:
def create_employees(employees_df, knowledge_instances):
  employees_df = employees_df.drop_duplicates()
  employee_knowledge = employees_df.groupby(['employee_id', 'full_name'])['knowledge_id'] \
  .apply(lambda x: list(set(x))).reset_index()

  employee_instances = {}
  for _, row in employee_knowledge.iterrows():
    # создадим сотрудника
      employee = Специалист(f"сотрудник_{row['employee_id']}")
      employee.имя_сотрудника = [row['full_name']]
      employee_instances[row['employee_id']] = employee

      # добавим знания сотруднику
      for knowledge_id in row['knowledge_id']:
          if knowledge_id in knowledge_instances:
              employee.обладает_знанием.append(knowledge_instances[knowledge_id])

  return employee_instances

In [20]:
employees = create_employees(employees_df, knowledges)

In [21]:
def print_employee_mindmap():
  print("\n" + "="*60)
  print("КАРТА ЗНАНИЙ КАЖДОГО СОТРУДНИКА")
  print("="*60)
  for employee_id, employee in employees.items():
          employee_name = employee.имя_сотрудника[0] if employee.имя_сотрудника else f"Сотрудник {employee_id}"

          print(f"\n{employee_name}:")

          # Получаем список знаний сотрудника
          knowledge_list = list(employee.обладает_знанием)

          if knowledge_list:
              print("Знания:")
              for knowledge in knowledge_list:
                  know_name = knowledge.название[0] if knowledge.название else knowledge.name
                  print(f"  - {know_name}")
          else:
              print("Знания: нет")

print_employee_mindmap()


КАРТА ЗНАНИЙ КАЖДОГО СОТРУДНИКА

Иванов Алексей Сергеевич:
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Петрова Мария Владимировна:
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Сидоров Дмитрий Петрович:
Знания:
  - Математическая статистика
  - Линейная алгебра
  - SQL и базы данных

Козлова Анна Игоревна:
Знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Федоров Максим Андреевич:
Знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Николаева Екатерина Дмитриевна:
Знания:
  - Визуализация данных
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Орлов Сергей Викторович:
Знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей


### Создание индивидуальностей для должностей и из связей с компетенциями


In [22]:
# Создаем экземпляры должностей
def create_position_instances(positions_df):
    position_instances = {}
    for _, row in positions_df.iterrows():
        position = Должность(f"должность_{row['position_id']}")
        position.name = row['position_name']
        position_instances[row['position_id']] = position
    return position_instances

In [23]:
positions = create_position_instances(positions_df)
positions

{1: ml_specialist_v1.ML Engineer,
 2: ml_specialist_v1.Data Scientist,
 3: ml_specialist_v1.Data Analyst,
 4: ml_specialist_v1.NLP Engineer,
 5: ml_specialist_v1.Computer Vision Engineer}

In [24]:
# Устанавливаем связи между должностями и компетенциями
def set_position_competency_relations(position_competency_df, position_instances, competency_instances):
    for _, row in position_competency_df.iterrows():
        position = position_instances[row['position_id']]
        competency = competency_instances[row['competency_id']]
        position.требует_компетенцию.append(competency)

In [25]:
set_position_competency_relations(position_competency_df, positions, competencies)

In [26]:
# сюда вставить код для выполнения самостоятельных заданий
# positions[4].требует_компетенцию.append(competencies[4])



## Выведем требования к должностям

In [27]:
# Проверим требования должностей
def print_position_require():
  print("\n" + "="*60)
  print("ТРЕБОВАНИЯ ДОЛЖНОСТЕЙ")
  print("="*60)

  for position_id, position in positions.items():
      pos_name = position.name
      print(f"\n{pos_name} требует компетенции:")

      required_competencies = list(position.требует_компетенцию)
      if required_competencies:
          for comp in required_competencies:
              print(f"  - {comp.название[0]}")
      else:
          print("  - нет требований")

print_position_require()


ТРЕБОВАНИЯ ДОЛЖНОСТЕЙ

ML Engineer требует компетенции:
  - Программирование и алгоритмы
  - Машинное обучение

Data Scientist требует компетенции:
  - Программирование и алгоритмы
  - Машинное обучение

Data Analyst требует компетенции:
  - Обработка и анализ данных

NLP Engineer требует компетенции:
  - Программирование и алгоритмы

Computer Vision Engineer требует компетенции:
  - Машинное обучение


## Создание аксиом для онтологии

 <b>Аксиома 1:</b> Если специалист обладает всеми требуемыми знаниями для компетенции, то у него есть эта компетенция

 <b>Аксиома 2:</b> Если у специалиста есть требуемые компетенции для должности, то он может занимать эту должность

  <b> Аксиома 3:</b> (для проверки корректности онтологии) Если специалист занимает должность, то у него должны быть соответствующие компетенции
      


In [28]:
def define_rules():
    # Аксиома 1:
    with onto:
        class ИмеетКомпетенциюПоЗнаниям(Специалист >> bool):
            def __init__(self):
                super().__init__()

            def __call__(self, specialist):
                # Для каждой компетенции проверяем, есть ли у специалиста все необходимые знания
                for competency in onto.Компетенция.instances():
                    required_knowledge = list(competency.требует_знание)
                    specialist_knowledge = list(specialist.обладает_знанием)

                    # Проверяем, что все требуемые знания есть у специалиста
                    if all(knowledge in specialist_knowledge for knowledge in required_knowledge):
                        if competency not in specialist.обладает_компетенцией:
                            specialist.обладает_компетенцией.append(competency)

                return True

        # Аксиома 2:
        class МожетЗаниматьДолжностьПоКомпетенциям(Специалист >> bool):
            def __init__(self):
                super().__init__()

            def __call__(self, specialist):
                # Для каждой должности проверяем, есть ли у специалиста все необходимые компетенции
                for position in onto.Должность.instances():
                    required_competencies = list(position.требует_компетенцию)
                    specialist_competencies = list(specialist.обладает_компетенцией)

                    # Проверяем, что все требуемые компетенции есть у специалиста
                    if all(competency in specialist_competencies for competency in required_competencies):
                        if position not in specialist.может_занимать_должность:
                            specialist.может_занимать_должность.append(position)

                return True

        # Аксиома 3:
        class ПроверитьКомпетенцииДляДолжности(Специалист >> bool):
            def __init__(self):
                super().__init__()

            def __call__(self, specialist):
                positions = list(specialist.может_занимать_должность)
                competencies = list(specialist.обладает_компетенцией)

                for position in positions:
                    required_competencies = list(position.требует_компетенцию)
                    if not all(comp in competencies for comp in required_competencies):
                        print(f"Предупреждение: у {specialist.name} нет всех компетенций для {position.name}")

                return True

    return ИмеетКомпетенциюПоЗнаниям(), МожетЗаниматьДолжностьПоКомпетенциям(), ПроверитьКомпетенцииДляДолжности()

In [29]:
rule1, rule2, rule3 = define_rules()

In [30]:
# Проверим связи знаний и компетенций
print_competency_knowledge()


СВЯЗИ ЗНАНИЙ И КОМПЕТЕНЦИЙ

Математическая подготовка требует знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей

Программирование и алгоритмы требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Машинное обучение требует знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Глубокое обучение требует знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Обработка и анализ данных требует знания:
  - Математическая статистика
  - SQL и базы данных

Работа с большими данными требует знания:
  - Работа с большими данными (Spark, Hadoop)
  - SQL и базы данных

Визуализация и представление данных требует знания:
  - Визуализация данных
  - Python программирование

Развертывание ML-моделей требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, 

In [31]:
onto.Специалист.instances()

[ml_specialist_v1.сотрудник_1,
 ml_specialist_v1.сотрудник_2,
 ml_specialist_v1.сотрудник_3,
 ml_specialist_v1.сотрудник_4,
 ml_specialist_v1.сотрудник_5,
 ml_specialist_v1.сотрудник_6,
 ml_specialist_v1.сотрудник_7]

In [32]:
for employee in onto.Специалист.instances():
        rule1(employee)  # Применяем правило для компетенций
        rule2(employee)  # Применяем правило для должностей
        rule3(employee)  # Проверяем соответствие

## Проведение анализа данных

In [33]:
def print_position_for_employee():
  print("\n" + "="*60)
  print("АНАЛИЗ ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ У СОТРУДНИКОВ")
  print("="*60)

  for employee in onto.Специалист.instances():
          print(f"\nСотрудник: {employee.имя_сотрудника[0]}")

          print("Знания:")
          for knowledge in employee.обладает_знанием:
              print(f"  - {knowledge.название[0]}")

          print("Компетенции:")
          for competency in employee.обладает_компетенцией:
              print(f"  - {competency.название[0]}")

          print("Может занимать должности:")
          for position in employee.может_занимать_должность:
              print(f"  - {position.name}")

          print("-" * 30)

print_position_for_employee()


АНАЛИЗ ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ У СОТРУДНИКОВ

Сотрудник: Иванов Алексей Сергеевич
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Компетенции:
  - Программирование и алгоритмы
  - Машинное обучение
  - Развертывание ML-моделей
Может занимать должности:
  - ML Engineer
  - Data Scientist
  - NLP Engineer
  - Computer Vision Engineer
------------------------------

Сотрудник: Петрова Мария Владимировна
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Компетенции:
  - Программирование и алгоритмы
  - Машинное обучение
  - Развертывание ML-моделей
Может занимать должности:
  - ML Engineer
  - Data Scientist
  - NLP Engineer
  - Computer Vision Engineer
------------------------------

Сотрудник: Сидоров Дмитрий Петрович
Знания:
  - Математическая статистика
  - Линейная алгебра
  - SQL и базы данных
Компетенции:
  - Обработка и анализ данных
Может зани

In [34]:
# Убедимся, что у сотрудников достаточно знаний для получения компетенций
def check_employee_competencies(employee_instances, competency_instances):
    """Проверяем, какие компетенции могут получить сотрудники"""
    print("\n" + "="*60)
    print("ПРОВЕРКА ВОЗМОЖНЫХ КОМПЕТЕНЦИЙ СОТРУДНИКОВ")
    print("="*60)

    for employee_id, employee in employee_instances.items():
        employee_name = employee.имя_сотрудника[0]
        print(f"\n{employee_name}:")

        for competency_id, competency in competency_instances.items():
            comp_name = competency.название[0]
            required_knowledge = set(competency.требует_знание)
            employee_knowledge = set(employee.обладает_знанием)

            # Проверяем, есть ли у сотрудника все необходимые знания
            if required_knowledge.issubset(employee_knowledge):
                print(f"  Может получить компетенцию: {comp_name}")
                # Присваиваем компетенцию
                if competency not in employee.обладает_компетенцией:
                    employee.обладает_компетенцией.append(competency)
            else:
                missing_knowledge = required_knowledge - employee_knowledge
                if missing_knowledge:
                    print(f"  Не хватает для '{comp_name}':")
                    for know in missing_knowledge:
                        print(f"      - {know.название[0]}")


In [35]:
check_employee_competencies(employees, competencies)


ПРОВЕРКА ВОЗМОЖНЫХ КОМПЕТЕНЦИЙ СОТРУДНИКОВ

Иванов Алексей Сергеевич:
  Не хватает для 'Математическая подготовка':
      - Теория вероятностей
      - Линейная алгебра
  Может получить компетенцию: Программирование и алгоритмы
  Может получить компетенцию: Машинное обучение
  Не хватает для 'Глубокое обучение':
      - Компьютерное зрение
      - Обработка естественного языка (NLP)
  Не хватает для 'Обработка и анализ данных':
      - SQL и базы данных
  Не хватает для 'Работа с большими данными':
      - Работа с большими данными (Spark, Hadoop)
      - SQL и базы данных
  Не хватает для 'Визуализация и представление данных':
      - Визуализация данных
  Может получить компетенцию: Развертывание ML-моделей

Петрова Мария Владимировна:
  Не хватает для 'Математическая подготовка':
      - Теория вероятностей
      - Линейная алгебра
  Может получить компетенцию: Программирование и алгоритмы
  Может получить компетенцию: Машинное обучение
  Не хватает для 'Глубокое обучение':
      -

In [36]:
def check_employee_positions(employee_instances, position_instances):
    """Проверяем, какие должности могут занять сотрудники"""
    print("\n" + "="*60)
    print("ПРОВЕРКА ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ СОТРУДНИКОВ")
    print("="*60)

    for employee_id, employee in employee_instances.items():
        employee_name = employee.имя_сотрудника[0]
        print(f"\n{employee_name}:")

        for position_id, position in position_instances.items():
            pos_name = position.name
            required_competencies = set(position.требует_компетенцию)
            employee_competencies = set(employee.обладает_компетенцией)

            # Проверяем, есть ли у сотрудника все необходимые компетенции
            if required_competencies.issubset(employee_competencies):
                print(f"   Может занять должность: {pos_name}")
                # Присваиваем должность
                if position not in employee.может_занимать_должность:
                    employee.может_занимать_должность.append(position)
            else:
                missing_competencies = required_competencies - employee_competencies
                if missing_competencies:
                    print(f"   Не хватает для '{pos_name}':")
                    for comp in missing_competencies:
                        print(f"      - {comp.название[0]}")

# Применяем проверки

check_employee_positions(employees, positions)



ПРОВЕРКА ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ СОТРУДНИКОВ

Иванов Алексей Сергеевич:
   Может занять должность: ML Engineer
   Может занять должность: Data Scientist
   Не хватает для 'Data Analyst':
      - Обработка и анализ данных
   Может занять должность: NLP Engineer
   Может занять должность: Computer Vision Engineer

Петрова Мария Владимировна:
   Может занять должность: ML Engineer
   Может занять должность: Data Scientist
   Не хватает для 'Data Analyst':
      - Обработка и анализ данных
   Может занять должность: NLP Engineer
   Может занять должность: Computer Vision Engineer

Сидоров Дмитрий Петрович:
   Не хватает для 'ML Engineer':
      - Машинное обучение
      - Программирование и алгоритмы
   Не хватает для 'Data Scientist':
      - Машинное обучение
      - Программирование и алгоритмы
   Может занять должность: Data Analyst
   Не хватает для 'NLP Engineer':
      - Программирование и алгоритмы
   Не хватает для 'Computer Vision Engineer':
      - Машинное обучение

Козлова Анна 

# Задание 1:  Добавление необходимой компетенции в должность

Добавить требование, что компетениция NLP Enginner содержит знание по глубокому обучению.

<B>Важно, что добавление в онтологию новых данных в Питоне должно сопровождаться созданием новой онтологии.</B>


In [37]:
# positions

In [38]:
# positions[4].требует_компетенцию

In [39]:
# competencies[2].название

In [41]:
# for i, item in competencies.items():
#  print (i, ' ', item.название)


In [42]:
#  positions[4].требует_компетенцию.append(competencies[4])

In [43]:
onto.save(file="ml_specialist_v1.owl", format="rdfxml")
print("✅ Исходная онтология сохранена как ml_specialist_v1.owl")

# 2. Загружаем её в новую онтологию
new_onto = get_ontology("ml_specialist_v1.owl").load()

# 3. Открываем новую онтологию для редактирования
with new_onto:
    
    # 4. Находим должность NLP Engineer (по имени или ID)
    nlp_position = None
    for pos in new_onto.Должность.instances():
        if hasattr(pos, 'name') and pos.name == "NLP Engineer":
            nlp_position = pos
            break
    
    # 5. Находим компетенцию "Глубокое обучение"
    deep_learning_comp = None
    for comp in new_onto.Компетенция.instances():
        if hasattr(comp, 'название') and comp.название and comp.название[0] == "Глубокое обучение":
            deep_learning_comp = comp
            break
    
    # 6. Добавляем требование
    if nlp_position and deep_learning_comp:
        if deep_learning_comp not in nlp_position.требует_компетенцию:
            nlp_position.требует_компетенцию.append(deep_learning_comp)
            print(f"✅ Добавлено: {nlp_position.name} требует {deep_learning_comp.название[0]}")
        else:
            print(f"⚠️ Уже есть: {nlp_position.name} требует {deep_learning_comp.название[0]}")
    
    # 7. Сохраняем новую онтологию
    new_onto.save(file="ml_specialist_v2.owl", format="rdfxml")
    print("✅ Новая онтология сохранена как ml_specialist_v2.owl")

# 8. Проверка
print("\n" + "="*50)
print(f"Должность: {nlp_position.name}")
print("Требуемые компетенции:")
for comp in nlp_position.требует_компетенцию:
    comp_name = comp.название[0] if hasattr(comp, 'название') and comp.название else "Без названия"
    print(f"  - {comp_name}")
print("="*50)

✅ Исходная онтология сохранена как ml_specialist_v1.owl
✅ Добавлено: NLP Engineer требует Глубокое обучение
✅ Новая онтология сохранена как ml_specialist_v2.owl

Должность: NLP Engineer
Требуемые компетенции:
  - Программирование и алгоритмы
  - Глубокое обучение


## Задание 2:
Добавьте требование в онтологию: должность Data Scientist требует компетенции "Математическая подготовка"

In [48]:
onto_v3 = get_ontology("ml_specialist_v2.owl").load()

with onto_v3:
    # Находим должность Data Scientist
    data_scientist = None
    for position in onto_v3.Должность.instances():
        if position.name == "Data Scientist":
            data_scientist = position
            break
    
    # Находим компетенцию "Математическая подготовка"
    math_competency = None
    for competency in onto_v3.Компетенция.instances():
        if competency.название[0] == "Математическая подготовка":
            math_competency = competency
            break
    
    # Добавляем требование
    if data_scientist and math_competency:
        if math_competency not in data_scientist.требует_компетенцию:
            data_scientist.требует_компетенцию.append(math_competency)
            print(f"✅ Добавлено: {data_scientist.name} требует {math_competency.название[0]}")
        else:
            print(f"⚠️ Требование уже существует")

# Сохраняем новую онтологию
onto_v3.save(file="ml_specialist_v3.owl", format="rdfxml")
print("✅ Новая онтология сохранена как ml_specialist_v3.owl")

# Проверка
print("\n" + "="*60)
print("ПРОВЕРКА ТРЕБОВАНИЙ ДОЛЖНОСТЕЙ В НОВОЙ ОНТОЛОГИИ")
print("="*60)

for position in onto_v3.Должность.instances():
    print(f"\n{position.name} требует компетенции:")
    for comp in position.требует_компетенцию:
        print(f"  - {comp.название[0]}")
print("="*60)

⚠️ Требование уже существует
✅ Новая онтология сохранена как ml_specialist_v3.owl

ПРОВЕРКА ТРЕБОВАНИЙ ДОЛЖНОСТЕЙ В НОВОЙ ОНТОЛОГИИ

ML Engineer требует компетенции:
  - Программирование и алгоритмы
  - Машинное обучение

Data Scientist требует компетенции:
  - Программирование и алгоритмы
  - Машинное обучение
  - Математическая подготовка

Data Analyst требует компетенции:
  - Обработка и анализ данных

NLP Engineer требует компетенции:
  - Программирование и алгоритмы
  - Глубокое обучение

Computer Vision Engineer требует компетенции:
  - Машинное обучение


## Задание 3:
Добавьте сотруднику Федорову Максиму Андреевичу необходимые знания, чтобы он мог претендовать на должность Computer Vision Engineer

In [49]:
onto_v4 = get_ontology("ml_specialist_v3.owl").load()

with onto_v4:
    # Находим сотрудника Федорова
    fedorov = None
    for employee in onto_v4.Специалист.instances():
        if employee.имя_сотрудника and employee.имя_сотрудника[0] == "Федоров Максим Андреевич":
            fedorov = employee
            break
    
    # Находим необходимые знания
    math_stat = None
    python_prog = None
    
    for knowledge in onto_v4.Знание.instances():
        if knowledge.название[0] == "Математическая статистика":
            math_stat = knowledge
        elif knowledge.название[0] == "Python программирование":
            python_prog = knowledge
    
    # Добавляем знания
    if fedorov and math_stat and python_prog:
        print(f"\nСотрудник: {fedorov.имя_сотрудника[0]}")
        print("Добавляем знания:")
        
        if math_stat not in fedorov.обладает_знанием:
            fedorov.обладает_знанием.append(math_stat)
            print(f"  + {math_stat.название[0]}")
        
        if python_prog not in fedorov.обладает_знанием:
            fedorov.обладает_знанием.append(python_prog)
            print(f"  + {python_prog.название[0]}")
    
    # Применяем правила для обновления компетенций и должностей
    # Создаём функции правил внутри новой онтологии
    class Rule1:
        def __call__(self, specialist):
            for comp in onto_v4.Компетенция.instances():
                required = list(comp.требует_знание)
                has_knowledge = list(specialist.обладает_знанием)
                if all(k in has_knowledge for k in required):
                    if comp not in specialist.обладает_компетенцией:
                        specialist.обладает_компетенцией.append(comp)
            return True
    
    class Rule2:
        def __call__(self, specialist):
            for pos in onto_v4.Должность.instances():
                required = list(pos.требует_компетенцию)
                has_comps = list(specialist.обладает_компетенцией)
                if all(c in has_comps for c in required):
                    if pos not in specialist.может_занимать_должность:
                        specialist.может_занимать_должность.append(pos)
            return True
    
    rule1 = Rule1()
    rule2 = Rule2()
    rule1(fedorov)
    rule2(fedorov)

# Сохраняем новую онтологию
onto_v4.save(file="ml_specialist_v4.owl", format="rdfxml")
print("\n✅ Новая онтология сохранена как ml_specialist_v4.owl")

# Проверка
print("\n" + "="*60)
print("РЕЗУЛЬТАТ ДЛЯ ФЕДОРОВА В НОВОЙ ОНТОЛОГИИ")
print("="*60)
print(f"\nСотрудник: {fedorov.имя_сотрудника[0]}")
print("Знания:")
for k in fedorov.обладает_знанием:
    print(f"  - {k.название[0]}")
print("Компетенции:")
for c in fedorov.обладает_компетенцией:
    print(f"  - {c.название[0]}")
print("Может занимать должности:")
for p in fedorov.может_занимать_должность:
    print(f"  - {p.name}")
print("="*60)


Сотрудник: Федоров Максим Андреевич
Добавляем знания:

✅ Новая онтология сохранена как ml_specialist_v4.owl

РЕЗУЛЬТАТ ДЛЯ ФЕДОРОВА В НОВОЙ ОНТОЛОГИИ

Сотрудник: Федоров Максим Андреевич
Знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение
  - Математическая статистика
  - Python программирование
Компетенции:
  - Глубокое обучение
  - Программирование и алгоритмы
  - Машинное обучение
  - Развертывание ML-моделей
Может занимать должности:
  - ML Engineer
  - NLP Engineer
  - Computer Vision Engineer


## Задание 4:
Добавьте требование в онтологию нового сотрудника, укажите в качестве имя_сотрудника ваше ФИО. Добавьте знания для данного сотрудника, чтобы он смог претендовать на все должности текущей онтологии

In [54]:
onto_v5 = get_ontology("ml_specialist_v4.owl").load()

with onto_v5:
    # Все необходимые знания
    required_knowledge_names = [
        "Математическая статистика",
        "Линейная алгебра",
        "Теория вероятностей",
        "Python программирование",
        "Библиотеки ML (scikit-learn, TensorFlow, PyTorch)",
        "Обработка естественного языка (NLP)",
        "Компьютерное зрение",
        "SQL и базы данных"
    ]
    
    # Собираем объекты знаний
    knowledge_objects = []
    for know_name in required_knowledge_names:
        for knowledge in onto_v5.Знание.instances():
            if knowledge.название[0] == know_name:
                knowledge_objects.append(knowledge)
                break
    
    employee_name = "Щепетова Мария Витальевна"
    
    # Создаём сотрудника
    new_employee = onto_v5.Специалист(f"сотрудник_{employee_name.replace(' ', '_')}")
    new_employee.имя_сотрудника = [employee_name]
    
    # Добавляем знания
    for knowledge in knowledge_objects:
        new_employee.обладает_знанием.append(knowledge)
    
    print(f"✅ Создан сотрудник: {employee_name}")
    print(f"✅ Добавлено знаний: {len(new_employee.обладает_знанием)}")
    
    # Применяем правила (те же, что в задании 3)
    class Rule1:
        def __call__(self, specialist):
            for comp in onto_v5.Компетенция.instances():
                required = list(comp.требует_знание)
                has_knowledge = list(specialist.обладает_знанием)
                if all(k in has_knowledge for k in required):
                    if comp not in specialist.обладает_компетенцией:
                        specialist.обладает_компетенцией.append(comp)
            return True
    
    class Rule2:
        def __call__(self, specialist):
            for pos in onto_v5.Должность.instances():
                required = list(pos.требует_компетенцию)
                has_comps = list(specialist.обладает_компетенцией)
                if all(c in has_comps for c in required):
                    if pos not in specialist.может_занимать_должность:
                        specialist.может_занимать_должность.append(pos)
            return True
    
    rule1 = Rule1()
    rule2 = Rule2()
    rule1(new_employee)
    rule2(new_employee)

# Сохраняем новую онтологию
onto_v5.save(file="ml_specialist_v5.owl", format="rdfxml")
print("\n✅ Новая онтология сохранена как ml_specialist_v5.owl")

# Проверка
print("\n" + "="*60)
print("РЕЗУЛЬТАТ ДЛЯ НОВОГО СОТРУДНИКА")
print("="*60)
print(f"\nСотрудник: {employee_name}")
print(f"\nКомпетенций получено: {len(new_employee.обладает_компетенцией)}")
print("Может занимать должности:")
for pos in new_employee.может_занимать_должность:
    print(f"  - {pos.name}")

total = len(list(onto_v5.Должность.instances()))
if len(new_employee.может_занимать_должность) == total:
    print(f"\n🎉 УСПЕХ! Все {total} должностей доступны!")
print("="*60)

✅ Создан сотрудник: Щепетова Мария Витальевна
✅ Добавлено знаний: 40

✅ Новая онтология сохранена как ml_specialist_v5.owl

РЕЗУЛЬТАТ ДЛЯ НОВОГО СОТРУДНИКА

Сотрудник: Щепетова Мария Витальевна

Компетенций получено: 6
Может занимать должности:
  - ML Engineer
  - Data Scientist
  - Data Analyst
  - NLP Engineer
  - Computer Vision Engineer

🎉 УСПЕХ! Все 5 должностей доступны!
